# Тема 10. Мультиагентні системи: Supervisor + Human-in-the-Loop

**Навчальний ноутбук, відредагована версія початкового уроку.**

Мета — зрозуміти, як **супервізор** делегує роботу двом спеціалізованим агентам: календарному та поштовому, і як додати **Human-in-the-Loop (HITL)** — рішення людини перед виконанням дії.

Після уроку ви зможете:
1. Розрізняти мовну модель, інструмент (`tool`), підагента та супервізора.
2. Побудувати мультиагентну систему на `LangChain` / `LangGraph` з OpenAI API.
3. Перевіряти поведінку агентів і бачити послідовність викликів.
4. Додати справжню паузу на **approve / edit / reject**.

> **Важливо.** Календар та email у цьому прикладі — **імітації (mock tools)**. Вони не підключені до Google Calendar, Outlook, Gmail чи SMTP і не створюють реальних подій / листів. **OpenAI API викликається насправді**, якщо ви виконаєте комірки з агентами; це може коштувати грошей.

## 0. Що саме будемо будувати

```text
Користувач
    │  Один запит: «Заплануй зустріч і надішли лист»
    ▼
Supervisor (LLM + два високорівневі tools)
    ├── schedule_event(request) ──► Calendar agent
    │                                 ├── get_available_time_slots
    │                                 └── create_calendar_event [MOCK]
    └── manage_email(request) ────► Email agent
                                      └── send_email [MOCK]
```

**LLM** — велика мовна модель (Large Language Model); вона вирішує, який інструмент викликати. **Підагент** — окремий агент з власним промптом та інструментами. **Supervisor** — агент-координатор, який викликає підагентів через інструменти.

У блоці HITL захищатимемо **високорівневі інструменти супервізора** (пауза перед делегуванням), а додатково покажемо затвердження **низькорівневого `send_email`** для редагування конкретного поля `subject`. Це **два різні рівні контролю**, і їх важливо не плутати.

## 1. Встановлення залежностей

Запустіть комірку один раз. `%pip` встановлює пакети у Python-середовище поточного Jupyter kernel. Для прикладу з `interrupt_mode="per_call"` потрібна версія `langchain >= 1.4.6`. Після інсталяції за потреби перезапустіть kernel.

**Середовище:** рекомендовано Python 3.11+. Потім виконуйте комірки згори вниз.

In [ ]:
%pip install -qU "langchain>=1.4.6" "langgraph>=1.0" langchain-openai

## 2. OpenAI API key (безпечне налаштування)

1. Якщо змінна середовища `OPENAI_API_KEY` вже встановлена — повторно нічого вводити не потрібно.
2. Інакше `getpass()` попросить ключ, **не відображаючи його в комірці**.
3. Ключ не слід записувати просто в код або публікувати разом із ноутбуком.

**LangSmith** — окремий, **необов'язковий** сервіс трасування. **LangSmith API key не потрібен**, щоб викликати OpenAI API. Тут трасування вимкнене за замовчуванням. Для його використання потрібен окремий ключ і усвідомлене ввімкнення.

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    api_key = getpass("Вставте OpenAI API key (ввід приховано): ").strip()
    if not api_key:
        raise ValueError("Потрібен непорожній OPENAI_API_KEY")
    os.environ["OPENAI_API_KEY"] = api_key
    del api_key

# Не вмикаємо трасування без потреби: це окремий сервіс із власним ключем.
ENABLE_LANGSMITH = False
if ENABLE_LANGSMITH:
    if not os.getenv("LANGSMITH_API_KEY"):
        langsmith_key = getpass("LangSmith API key (ввід приховано): ").strip()
        if not langsmith_key:
            raise ValueError("Для LangSmith потрібен LANGSMITH_API_KEY")
        os.environ["LANGSMITH_API_KEY"] = langsmith_key
        del langsmith_key
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "learning-multiagents")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    os.environ["LANGCHAIN_TRACING_V2"] = "false"

print("OpenAI API key налаштовано (значення не показуємо).")
print("LangSmith tracing:", os.environ["LANGSMITH_TRACING"])

## 3. Імпорти та робочі дані прикладу

Імпортуємо інструменти LangChain, об'єкт OpenAI chat-моделі, засоби роботи з часом та генератор унікального ID для окремого HITL-сеансу.

Для демонстрації використовуємо **вигадані тестові адреси `@example.com`**, щоб приклад не містив реальних персональних даних. Дата «наступного вівторка» обчислюється від поточного дня у часовому поясі `Europe/Warsaw`; час у ISO 8601 матиме правильне зміщення, зокрема з урахуванням літнього часу.

In [ ]:
from datetime import date as date_type, datetime, time, timedelta
from uuid import uuid4
from zoneinfo import ZoneInfo

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

TIMEZONE = ZoneInfo("Europe/Warsaw")
TODAY = datetime.now(TIMEZONE).date()
days_until_next_tuesday = (1 - TODAY.weekday()) % 7 or 7
MEETING_DATE = TODAY + timedelta(days=days_until_next_tuesday)
MEETING_START = datetime.combine(MEETING_DATE, time(14, 0), tzinfo=TIMEZONE)
MEETING_END = MEETING_START + timedelta(hours=1)
TEST_RECIPIENTS = ["designer1@example.com", "designer2@example.com"]

print("Дата зустрічі:", MEETING_DATE)
print("Початок (ISO 8601):", MEETING_START.isoformat())
print("Тестові адреси:", ", ".join(TEST_RECIPIENTS))

## 4. Mock-інструменти: чим є та чим НЕ є ці функції

**Tool** — функція, схему аргументів якої бачить агент. Декоратор `@tool` перетворює Python-функцію на інструмент. Назва, type hints та **docstring** допомагають моделі зрозуміти, коли й як його використовувати.

Повернення рядка «створено» **не означає**, що відбувся зовнішній API-виклик. Щоб результати були перевірюваними, mock-інструменти записуватимуть дії до списків `SIMULATED_EVENTS` і `SIMULATED_EMAILS` у пам'яті ноутбука.

In [ ]:
SIMULATED_EVENTS: list[dict] = []
SIMULATED_EMAILS: list[dict] = []

print("Mock-журнали створено; спочатку вони порожні.")

### 4.1. `create_calendar_event` — створення події (імітація)

**Вхід:** назва, час початку/завершення у ISO 8601 **з часовим поясом**, учасники та необов'язкове місце.

**Перевірка:** завершення пізніше початку, учасники задані, дата і час мають часовий пояс. Якщо дані некоректні, `ValueError` сигналізує про помилку; інакше подія потрапляє тільки в mock-журнал.

In [ ]:
@tool
def create_calendar_event(
    title: str,
    start_time: str,
    end_time: str,
    attendees: list[str],
    location: str = "",
) -> str:
    """Імітувати створення календарної події без зовнішнього API.

    Args:
        title: Назва події.
        start_time: Початок у ISO 8601 із часовим поясом, наприклад
            '2026-11-10T14:00:00+01:00'.
        end_time: Завершення у ISO 8601 із часовим поясом.
        attendees: Список email-адрес учасників події.
        location: Необов'язкове місце проведення.

    Returns:
        Повідомлення з чіткою позначкою, що подія лише симульована.

    Raises:
        ValueError: Якщо час некоректний або учасники не вказані.
    """
    start = datetime.fromisoformat(start_time.replace("Z", "+00:00"))
    end = datetime.fromisoformat(end_time.replace("Z", "+00:00"))
    if start.tzinfo is None or end.tzinfo is None:
        raise ValueError("start_time і end_time мають містити часовий пояс")
    if end <= start:
        raise ValueError("end_time має бути пізніше start_time")
    if not attendees:
        raise ValueError("Потрібно вказати принаймні одного учасника")

    record = {
        "title": title,
        "start_time": start.isoformat(),
        "end_time": end.isoformat(),
        "attendees": list(attendees),
        "location": location,
    }
    SIMULATED_EVENTS.append(record)
    return f"[MOCK] Подію '{title}' записано лише до SIMULATED_EVENTS."

### 4.2. `send_email` — надсилання листа (імітація)

**Вхід:** одержувачі, тема, текст листа, необов'язкові копії (CC — Carbon Copy).

Виправлення щодо початкового коду: замість `cc=[]` використано `cc=None` — так ми уникаємо **змінюваного аргументу за замовчуванням** (mutable default argument). Список створюється всередині функції. Навіть після виклику цього інструмента лист **не буде відправлено**.

In [ ]:
@tool
def send_email(
    to: list[str],
    subject: str,
    body: str,
    cc: list[str] | None = None,
) -> str:
    """Імітувати надсилання email та зберегти запис у пам'яті.

    Args:
        to: Список email-адрес основних одержувачів.
        subject: Тема електронного листа.
        body: Текст листа.
        cc: Необов'язковий список одержувачів копії.

    Returns:
        Повідомлення, яке прямо зазначає відсутність реальної доставки.

    Raises:
        ValueError: Якщо немає адресатів чи теми або адреси некоректні.
    """
    recipients_cc = list(cc or [])
    if not to:
        raise ValueError("Потрібен хоча б один одержувач")
    if not subject.strip():
        raise ValueError("Тема листа не може бути порожньою")
    if not all("@" in address for address in [*to, *recipients_cc]):
        raise ValueError("Деякі адреси не схожі на email")

    record = {
        "to": list(to),
        "cc": recipients_cc,
        "subject": subject,
        "body": body,
    }
    SIMULATED_EMAILS.append(record)
    return f"[MOCK] Email до {', '.join(to)} НЕ надіслано; його записано лише до SIMULATED_EMAILS."

### 4.3. `get_available_time_slots` — перевірка слотів (імітація)

**Вхід:** учасники, дата в ISO-форматі `YYYY-MM-DD`, тривалість у хвилинах.

Цей інструмент **не опитує календар**: щоразу повертає заздалегідь задані години. Це корисно для навчання патерну агентів, але **не є реальною перевіркою доступності**.

In [ ]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,
    duration_minutes: int,
) -> list[str]:
    """Повернути демонстраційні часові слоти без запиту до календаря.

    Args:
        attendees: Список учасників для умовної перевірки доступності.
        date: Дата у форматі ISO 8601, 'YYYY-MM-DD'.
        duration_minutes: Бажана тривалість події у хвилинах.

    Returns:
        Статичний список демонстраційних годин: 09:00, 14:00, 16:00.

    Raises:
        ValueError: Якщо немає учасників, дата або тривалість некоректні.
    """
    date_type.fromisoformat(date)
    if not attendees:
        raise ValueError("Не вказані учасники")
    if duration_minutes <= 0:
        raise ValueError("Тривалість має бути додатною")
    return ["09:00", "14:00", "16:00"]

### 4.4. Швидкий тест інструмента без LLM

Можна перевірити mock-функції **до запуску моделі**. `.invoke({...})` передає словник аргументів інструмента. Це **не витрачає токени OpenAI**. Нижче тестуємо лише перевірку слотів; журнал подій та листів залишається порожнім.

In [ ]:
slots = get_available_time_slots.invoke(
    {
        "attendees": TEST_RECIPIENTS,
        "date": MEETING_DATE.isoformat(),
        "duration_minutes": 60,
    }
)
assert slots == ["09:00", "14:00", "16:00"]
print("Тест успішний; mock-слоти:", slots)

## 5. Підключаємо OpenAI chat-модель

`ChatOpenAI` із `langchain-openai` використовує `OPENAI_API_KEY` зі змінної середовища. Для навчального прикладу типовою є `gpt-4o-mini` — за потреби змініть `OPENAI_MODEL` **перед виконанням** цієї комірки.

`temperature=0` зазвичай зменшує випадковість відповідей, але **не гарантує повністю детерміновану поведінку**. Кожен виклик агента може породжувати кілька API-запитів (і відповідну оплату).

In [ ]:
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
model = ChatOpenAI(model=MODEL_NAME, temperature=0)
print("Налаштовано OpenAI-модель:", MODEL_NAME)

## 6. Створюємо спеціалізованих підагентів

### 6.1. `calendar_agent`

`create_agent` формує цикл **модель → виклик інструмента → результат інструмента → модель**. Календарний агент бачить **лише свої** інструменти: перевірку слотів і створення події.

В інструкції вимагаємо явні дані про учасників і час. Якщо даних немає, агент має **уточнити**, а не вигадати email чи дату. У цьому уроці всі календарні операції залишаються mock.

In [ ]:
calendar_agent = create_agent(
    model=model,
    tools=[get_available_time_slots, create_calendar_event],
    system_prompt=(
        "Ти календарний агент. Працюєш з датами ISO 8601 і часовими поясами. "
        "Не вигадуй email-адреси, дату, час чи доступність. "
        "Якщо бракує даних — постав уточнювальне запитання. "
        "За наявності даних спочатку викликай get_available_time_slots; "
        "якщо потрібного часу немає в списку, не викликай create_calendar_event. "
        "Усі інструменти демонстраційні: називай створення події імітацією."
    ),
)
print("Календарного агента створено.")

### 6.2. `email_agent`

Поштовий агент має тільки `send_email`, тому не може сам створювати події календаря. Його відповідальність — скласти лист із переданої інформації. Відсутні адреси потрібно запросити у користувача, а не вгадувати.

In [ ]:
email_agent = create_agent(
    model=model,
    tools=[send_email],
    system_prompt=(
        "Ти агент електронної пошти. Готуй короткі ввічливі повідомлення. "
        "Не вигадуй одержувачів, їхні адреси чи факти. Якщо бракує адрес — запитай. "
        "За наявності всіх даних викликай send_email. "
        "Підкреслюй, що інструмент лише імітує відправку листів."
    ),
)
print("Поштового агента створено.")

## 7. Підагент як інструмент супервізора

Щоб супервізор не знав про дрібні деталі календаря чи листа, «загортаємо» кожного підагента у **високорівневий tool** із параметром `request`.

Кожен виклик виконує **новий короткий діалог підагента**, передає йому задачу, а супервізору повертає лише останню відповідь. Це допомагає **ізолювати контекст**. Docstring описує делеговану компетенцію.

In [ ]:
@tool
def schedule_event(request: str) -> str:
    """Делегувати календарну задачу спеціалізованому календарному агенту.

    Args:
        request: Природномовне завдання з відомими датою, часом,
            учасниками і часовим поясом або запит на уточнення цих даних.

    Returns:
        Фінальна відповідь календарного підагента. Операції є mock.
    """
    result = calendar_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    content = result["messages"][-1].content
    return content if isinstance(content, str) else str(content)


@tool
def manage_email(request: str) -> str:
    """Делегувати підготовку mock-листа спеціалізованому email-агенту.

    Args:
        request: Текст задачі з адресами, метою листа і контекстом.

    Returns:
        Фінальна відповідь поштового підагента; реальної доставки немає.
    """
    result = email_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    content = result["messages"][-1].content
    return content if isinstance(content, str) else str(content)

print("Створено два високорівневі tools: schedule_event, manage_email.")

## 8. `supervisor_agent` — координатор двох підагентів

Модель супервізора бачить **тільки** `schedule_event` і `manage_email`. Вона визначає, кому делегувати частини користувацького запиту. У промпті просимо **спершу календар**, а потім email, бо другий крок змістовно залежить від першого.

> **Обмеження:** системний промпт впливає на вибір агента, але сам по собі не дає суворої гарантії порядку. Для жорсткої послідовності у production потрібен явно визначений workflow (наприклад, вузли й переходи LangGraph).

In [ ]:
SUPERVISOR_PROMPT = (
    "Ти супервізор календарного і поштового агентів. "
    "За потреби делегуй календарні задачі інструменту schedule_event, "
    "а електронну пошту — інструменту manage_email. "
    "Для запиту 'створи подію та надішли нагадування' спочатку спробуй "
    "отримати результат календарного агента, потім делегуй підготовку листа. "
    "Не вигадуй дані, яких немає. "
    "Пам'ятай, обидва зовнішні сервіси тут лише імітуються."
)

supervisor_agent = create_agent(
    model=model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)
print("Супервізора створено.")

## 9. Спостерігаємо за виконанням агентів

**Streaming** повертає оновлення з вузлів агентного графа. `stream_mode="updates"` зручний для навчання: можна бачити повідомлення моделі, tool calls та результати. Декілька допоміжних рядків оформимо в окрему функцію з docstring, щоб не дублювати цикли в кожному тесті.

**Важливо:** внутрішні повідомлення підагента не обов'язково повністю видимі у стрімі *супервізора* — підагент виконується всередині tool. Для його деталей є окремі тести нижче або LangSmith tracing.

In [ ]:
def print_agent_stream(agent, user_request: str) -> None:
    """Виконати агентний запит і надрукувати оновлення вузлів графа.

    Args:
        agent: Агент, створений через LangChain `create_agent`.
        user_request: Задача користувача природною мовою.

    Returns:
        None. Повідомлення друкуються безпосередньо у ноутбуці.

    Note:
        Це справжній виклик OpenAI API; він не є безкоштовним mock-тестом.
    """
    for update in agent.stream(
        {"messages": [{"role": "user", "content": user_request}]},
        stream_mode="updates",
    ):
        for node_name, state_update in update.items():
            print(f"\n--- Вузол: {node_name} ---")
            if isinstance(state_update, dict):
                for message in state_update.get("messages", []):
                    if hasattr(message, "pretty_print"):
                        message.pretty_print()
                    else:
                        print(message)

print("Допоміжна функція для демонстрацій готова.")

### 9.1. Тест повного сценарію: supervisor → два підагенти

Передаємо **однозначну дату, часовий пояс і тестові email**, щоб модель не мусила домислювати відсутні дані. Спостерігаємо, які інструменти вибере супервізор і що вони повернуть.

> Запуск цієї та наступних agent-комірок **здійснює реальні OpenAI API-запити**. Результат може змінюватись між запусками.

In [ ]:
user_request = (
    f"Створи демонстраційну зустріч команди дизайну {MEETING_START.isoformat()} "
    f"до {MEETING_END.isoformat()} з учасниками {', '.join(TEST_RECIPIENTS)}. "
    "Перевір умовні вільні слоти, потім підготуй mock-email цим учасникам "
    "з нагадуванням про перегляд нових макетів. Не надсилай реальні листи."
)

print("Запит користувача:", user_request)
print_agent_stream(supervisor_agent, user_request)

### 9.2. Тест календарного агента окремо

Тестуючи підагента без супервізора, ми ізолюємо проблему: якщо він неправильно перетворює дату або викликає tool, помилку легше локалізувати. Повторний запуск може додати ще одну **mock-подію** до журналу.

In [ ]:
query_calendar = (
    f"Створи демонстраційну зустріч з {', '.join(TEST_RECIPIENTS)} "
    f"{MEETING_START.isoformat()}–{MEETING_END.isoformat()}; "
    "спочатку перевір mock-слоти."
)
print_agent_stream(calendar_agent, query_calendar)

### 9.3. Тест поштового агента окремо

Перевіряємо, чи підготує агент тему, зміст і виклик `send_email`, коли адреси вже відомі. Результат — **тільки запис у пам'яті**.

In [ ]:
query_email = (
    f"Підготуй mock-email для {', '.join(TEST_RECIPIENTS)} про перегляд "
    f"нових макетів на зустрічі {MEETING_START.isoformat()}. "
    "Доставка не потрібна: використай лише демонстраційний інструмент."
)
print_agent_stream(email_agent, query_email)

### 9.4. Перевірка фактичних результатів mock-інструментів

Ми **не покладаємося тільки на текст відповіді LLM**. Навіть якщо агент написав «успішно», в тесті перевіряємо **спостережувану дію** у списках. Це важлива різниця між `assert` / фактичним ефектом і самозвітом моделі.

> Журнали накопичують дані при кожному виконанні, тому після повторних запусків кількість записів зростає.

In [ ]:
print("Кількість mock-подій:", len(SIMULATED_EVENTS))
print("Кількість mock-листів:", len(SIMULATED_EMAILS))
if SIMULATED_EVENTS:
    print("Остання подія:", SIMULATED_EVENTS[-1])
if SIMULATED_EMAILS:
    print("Останній лист:", SIMULATED_EMAILS[-1])

## 10. Human-in-the-Loop (HITL): затвердження делегування

**Проблема початкової версії:** middleware з interrupt було додано **всередину підагентів**, але checkpointer — тільки супервізору. Крім того, код очікував стару/несумісну форму даних (`tool`, `arguments`) і **автоматично затверджував дії**, не звертаючись до людини.

**Виправлення для навчального прикладу:** поставимо `HumanInTheLoopMiddleware` на **високорівневі tools супервізора** `schedule_event` і `manage_email`. Так супервізор зробить паузу **перед** делегуванням. Це **не замінює** перевірок у реальних календарних/email API — тут ми навчаємося керувати паузами на рівні супервізора.

Потрібні:
- `checkpointer=InMemorySaver()` — пам'ять стану **до перезапуску kernel**;
- `thread_id` — ID конкретного діалогу;
- `interrupt_mode="per_call"` — окреме переривання для кожного інструмента (`langchain >= 1.4.6`);
- `Command(resume=...)` — продовження **після рішення людини**.

Можливі рішення: `approve` (виконати), `edit` (змінити аргументи), `reject` (пропустити). Ми **не дозволяємо** `respond` для інструментів з ефектами.

### 10.1. Створюємо HITL-супервізора

Це новий агент із тим самим `SUPERVISOR_PROMPT`, але додатковим middleware. Обидва високорівневі інструменти потребують дозволу. Після схвалення підагент виконає свої **mock-інструменти**.

In [ ]:
supervisor_agent_hitl = create_agent(
    model=model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "schedule_event": {"allowed_decisions": ["approve", "edit", "reject"]},
                "manage_email": {"allowed_decisions": ["approve", "edit", "reject"]},
            },
            interrupt_mode="per_call",
            description_prefix="Потрібне рішення людини перед делегуванням",
        )
    ],
    checkpointer=InMemorySaver(),
)
print("HITL-супервізора створено.")

### 10.2. Запускаємо запит **до першої паузи**

`version="v2"` повертає об'єкт з полями `value` і `interrupts`; так легше відрізнити завершення від очікування підтвердження. `thread_id` створюємо заново, щоб повторне виконання не підхопило попередню незавершену сесію.

**У цій комірці ми НЕ підтверджуємо нічого.** Якщо модель спробує викликати контрольований tool, її виконання зупиниться. Показуємо назву запропонованої дії, параметри та її ID.

In [ ]:
hitl_query = user_request
hitl_config = {"configurable": {"thread_id": f"supervisor-{uuid4()}"}}

hitl_result = supervisor_agent_hitl.invoke(
    {"messages": [{"role": "user", "content": hitl_query}]},
    config=hitl_config,
    version="v2",
)

print("Кількість переривань:", len(hitl_result.interrupts))
for interrupt_ in hitl_result.interrupts:
    print("\nID:", interrupt_.id)
    print("Назва tool:", interrupt_.value.get("name"))
    print("Аргументи:", interrupt_.value.get("args"))

if not hitl_result.interrupts:
    print("Модель не запропонувала контрольованого tool; перевірте її відповідь:")
    print(hitl_result.value["messages"][-1].content)

### 10.3. Функція ручного рев'ю: approve / edit / reject

Цей крок **справді запитує людину** через `input()`, а не схвалює все автоматично. `edit` дозволяє змінити **існуючий аргумент** (наприклад, `request` на рівні супервізора, або `subject` у прямому `send_email`).

Під час редагування типи аргументів залишаються сумісними (`list`, `int`, `str`). Незнайоме поле додавати не можна; назва інструмента залишається тією самою. Якщо обрати `reject`, інструмент **не запуститься**. Введення неправильного рішення припинить процес із помилкою — це безпечніше, ніж мовчазно схвалити.

In [ ]:
def collect_human_decisions(interrupts) -> dict[str, dict]:
    """Зібрати явне рішення користувача для кожного HITL-переривання.

    Args:
        interrupts: Переривання LangGraph у режимі `per_call`. Кожне має
            `id` і `value` з назвою tool та первинними аргументами.

    Returns:
        Словник `{interrupt_id: decision}` для `Command(resume=...)`.

    Raises:
        ValueError: Якщо рішення, поле редагування або нове значення некоректні.

    Note:
        Жодна дія не схвалюється автоматично. Людина має обрати
        `approve`, `edit` або `reject` для КОЖНОГО переривання.
    """
    decisions: dict[str, dict] = {}
    for interrupt_ in interrupts:
        action = interrupt_.value
        action_name = action["name"]
        original_args = dict(action["args"])

        print("\n=== Запит дозволу ===")
        print("Tool:", action_name)
        print("Аргументи:", original_args)
        choice = input("Рішення [approve/edit/reject]: ").strip().lower()

        if choice == "approve":
            decision = {"type": "approve"}
        elif choice == "reject":
            reason = input("Причина відхилення: ").strip() or "Не затверджено користувачем"
            decision = {"type": "reject", "message": reason}
        elif choice == "edit":
            field = input(f"Поле для редагування {list(original_args)}: ").strip()
            if field not in original_args:
                raise ValueError(f"Tool не приймає поля {field!r}")
            original_value = original_args[field]
            raw = input(f"Нове значення поля {field}: ").strip()
            if not raw:
                raise ValueError("Нове значення не може бути порожнім")

            if isinstance(original_value, list):
                new_value = [item.strip() for item in raw.split(",") if item.strip()]
            elif isinstance(original_value, int) and not isinstance(original_value, bool):
                new_value = int(raw)
            elif isinstance(original_value, str):
                new_value = raw
            else:
                raise ValueError("Цей тип аргументу не підтримує просте редагування")

            edited_args = {**original_args, field: new_value}
            decision = {
                "type": "edit",
                "edited_action": {"name": action_name, "args": edited_args},
            }
        else:
            raise ValueError("Оберіть лише approve, edit або reject")

        decisions[interrupt_.id] = decision

    return decisions

print("Функцію ручного рев'ю визначено.")

### 10.4. Продовжуємо виконання **лише після вашого рішення**

Ця комірка питає дозвіл для кожного переривання і передає рішення у `Command(resume=...)`. Система може зупинятися **декілька разів**: наприклад, спершу перед календарем, потім перед email. Ми повторюємо цикл до завершення або до 6 раундів рев'ю — це захист від нескінченного навчального циклу.

**Порада:** спробуйте спочатку `reject`, потім на новому запуску `approve` чи `edit`. Тут усі зовнішні дії залишаються mock.

In [ ]:
for review_round in range(1, 7):
    if not hitl_result.interrupts:
        print("Роботу завершено без нових переривань.")
        break

    print(f"\n--- HITL: раунд {review_round} ---")
    decisions = collect_human_decisions(hitl_result.interrupts)
    hitl_result = supervisor_agent_hitl.invoke(
        Command(resume=decisions),
        config=hitl_config,  # Той самий thread_id — критично важливо.
        version="v2",
    )
else:
    print("Досягнуто ліміт раундів рев'ю; виконання ще може бути призупиненим.")

if not hitl_result.interrupts:
    print("\nФінальна відповідь супервізора:")
    print(hitl_result.value["messages"][-1].content)
else:
    print("Залишилися переривання:", len(hitl_result.interrupts))

## 11. Додатковий приклад: редагування **саме теми листа**

У початковому ноутбуці код намагався змінити `subject` у перериванні, де супервізор бачить лише `request`. Це різні схеми аргументів!

Щоб змінювати **структурований `subject`**, треба перехопити **конкретний tool `send_email`**, у якого справді є аргумент `subject`. Для цього тут окремий `email_approval_agent` з middleware та **власним checkpointer**.

Цей приклад незалежний від попереднього HITL-супервізора. Він **не є** автоматичним перехопленням внутрішнього виклику підагента з розділу 10. Для production-архітектури з HITL на різних рівнях необхідно окремо проєктувати передавання контексту й checkpointing між графами.

### 11.1. Створюємо окремого поштового агента з перевіркою `send_email`

Middleware затримає виклик `send_email` **до** запису mock-листа в журнал. Тільки після вашого рішення виконання продовжиться.

In [ ]:
email_approval_agent = create_agent(
    model=model,
    tools=[send_email],
    system_prompt=(
        "Ти поштовий агент. Якщо є точні email адреси, склади лист і виклич send_email. "
        "Інструмент send_email демонстраційний, реальних листів не надсилає. "
        "Не вигадуй одержувачів і факти."
    ),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {"allowed_decisions": ["approve", "edit", "reject"]}
            },
            interrupt_mode="per_call",
        )
    ],
    checkpointer=InMemorySaver(),
)
print("Email-агента з перевіркою аргументів створено.")

### 11.2. Запускаємо підготовку листа і переглядаємо його поля

Під час зупинки знайдіть у `args` три основні параметри: `to`, `subject`, `body`. Поки дія призупинена, mock-лист ще не потрапив до журналу.

In [ ]:
email_hitl_config = {"configurable": {"thread_id": f"email-{uuid4()}"}}
emails_before_review = len(SIMULATED_EMAILS)
email_hitl_query = (
    f"Підготуй mock-лист для {', '.join(TEST_RECIPIENTS)} із темою "
    "'Нагадування про нові макети' і коротким нагадуванням про перегляд "
    f"макетів {MEETING_START.isoformat()}."
)

email_hitl_result = email_approval_agent.invoke(
    {"messages": [{"role": "user", "content": email_hitl_query}]},
    config=email_hitl_config,
    version="v2",
)

for interrupt_ in email_hitl_result.interrupts:
    print("Tool:", interrupt_.value["name"])
    print("Аргументи:", interrupt_.value["args"])

### 11.3. Відредагуйте `subject` і підтвердьте

Виконайте комірку; якщо модель запропонувала `send_email`, оберіть **`edit`**, потім поле **`subject`**, далі введіть нову тему, наприклад «ВАЖЛИВО: нагадування про макети». Тоді буде записаний **змінений mock-лист**, а не початковий.

Ви також можете обрати `reject` і впевнитися, що новий лист не з'явиться в `SIMULATED_EMAILS`.

In [ ]:
for review_round in range(1, 5):
    if not email_hitl_result.interrupts:
        break
    email_decisions = collect_human_decisions(email_hitl_result.interrupts)
    email_hitl_result = email_approval_agent.invoke(
        Command(resume=email_decisions),
        config=email_hitl_config,
        version="v2",
    )

if not email_hitl_result.interrupts:
    print("Фінальна відповідь:")
    print(email_hitl_result.value["messages"][-1].content)
else:
    print("Залишилися переривання:", len(email_hitl_result.interrupts))

if len(SIMULATED_EMAILS) > emails_before_review:
    print("Тема нового mock-листа:", SIMULATED_EMAILS[-1]["subject"])
else:
    print("Нових mock-листів у цьому сценарії не створено.")

## 12. QA-перевірки та типові ризики

Розглянемо систему не лише як LLM-демо, а і як **об'єкт тестування**.

| Що перевіряємо | Очікувана поведінка | Чому це важливо |
| --- | --- | --- |
| Відсутні email-адреси | Агент уточнює, не вигадує адресатів | Безпека і коректність даних |
| Невалідний ISO / часовий пояс | Tool відхиляє ввід | Коректність планування |
| Завершення раніше початку | Tool відхиляє ввід | Бізнес-правило |
| `reject` в HITL | Контрольований tool не запускається | Дійсний людський контроль |
| `edit` в HITL | Викликається tool зі зміненими аргументами | Правильність редагування |
| Повторний запуск ноутбука | У mock-журналах можливі дублікати | Idempotency, retry та побічні ефекти |
| `stream` і фінальний текст | Текст LLM може не відповідати фактичним діям | Перевіряємо результати tools, а не лише відповідь |
| Відключений LangSmith | OpenAI API працює без LangSmith | Незалежність сервісів |
| Втрата kernel | `InMemorySaver` втрачає стан | Для production потрібен persistent storage |

**Обмеження прикладу:** немає реальних інтеграцій, зберігання даних між перезапусками, строгого orchestration порядку, захисту від повторних дій, тестів на збій API та повної перевірки адрес. Це навчальна демонстрація, а **не production-ready застосунок**.

### 12.1. Невелика перевірка негативного сценарію без витрат OpenAI

Навмисно передаємо некоректний діапазон часу до mock-інструмента. Очікуємо `ValueError` і перевіряємо, що журнал подій **не змінився**.

In [ ]:
events_before = len(SIMULATED_EVENTS)
try:
    create_calendar_event.invoke(
        {
            "title": "Некоректний тест",
            "start_time": MEETING_END.isoformat(),
            "end_time": MEETING_START.isoformat(),
            "attendees": TEST_RECIPIENTS,
        }
    )
except ValueError as error:
    print("Очікувана помилка:", error)
else:
    raise AssertionError("Некоректна подія не була відхилена")

assert len(SIMULATED_EVENTS) == events_before
print("PASS: невірна подія не записана.")

## 13. Самостійні вправи

1. **Початковий рівень.** Змініть запит на одну дію (лише календар). Які tools викликає супервізор?
2. **Середній рівень.** Приберіть адреси з `user_request`. Чи уточнює агент дані? Перевірте `SIMULATED_EMAILS`.
3. **Середній рівень.** Під час HITL-рев'ю спробуйте `approve`, `edit` та `reject`. Чи відповідає кожне рішення фактичним записам у журналі?
4. **Просунутий рівень.** Додайте в mock-подію унікальний `event_id` і захист від дублювання за ним. Напишіть негативний тест.
5. **Просунутий рівень.** Замініть промптове «спочатку календар, потім email» на **явний граф із послідовними вузлами**, щоб порядок був гарантованим.

**Питання для самоперевірки:** Чим відрізняється затвердження `manage_email(request=...)` на рівні супервізора від затвердження `send_email(to=..., subject=..., body=...)` всередині email-агента?

## 14. Висновки

У цьому уроці ми реалізували **патерн супервізора**:
1. **Низькорівневі mock-tools** — умовні календарні операції та email із перевірками вхідних даних.
2. **Підагентів** — календарного та поштового, кожного зі своїми інструментами й системним промптом.
3. **Супервізора** — делегування задач через `schedule_event` та `manage_email`.
4. **HITL** — реальні переривання з `checkpointer`, `thread_id` і **ручними** рішеннями `approve`, `edit`, `reject`.
5. **Тестування ефектів** — перевірку записів mock-журналів замість довіри до заяв моделі.

**Зверніть увагу:** OpenAI API тут використовується для LLM-викликів; LangSmith є необов'язковим. Жоден справжній лист чи календарна подія не створюються.

### Офіційні джерела
- [LangChain — Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents)
- [LangChain — Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangChain — OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai)
- [LangGraph — Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [OpenAI — API keys](https://platform.openai.com/api-keys)

*Навчальний матеріал. Поведінка LLM та інтерфейси бібліотек можуть змінюватися між версіями.*